<a href="https://colab.research.google.com/github/GalardOnly/GalardOnly/blob/main/Engenharia_de_Atributos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
dataset = pd.read_csv('credit_simple.csv',sep=';')
dataset.shape

(1000, 8)

In [ ]:
dataset.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,CLASSE
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv,bom
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv,ruim
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação,bom
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento,bom
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo,ruim


In [ ]:
y = dataset['CLASSE']
x = dataset.iloc[:,:-1]

In [ ]:
x.isnull().sum() #aqui vemos qual colunas tem valores nulos, e podemos colocar a mediana no lugar

,0
SALDO_ATUAL,7
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


In [ ]:
mediana = x['SALDO_ATUAL'].median() #aqui calculamos a mediana
mediana

2323.0

In [ ]:
x['SALDO_ATUAL'].fillna(mediana, inplace=True) #aqui a gente coloca a mediana no saldo atual e conferimos se deu certo
x.isnull().sum()

<ipython-input-9-3a65aed8c68e>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  x['SALDO_ATUAL'].fillna(mediana, inplace=True)


,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


In [ ]:
agrupado = x.groupby(['ESTADOCIVIL']).size() #aqui escolhemos a moda como metodo e conferimos qual sera
agrupado #descobrimos que a moda é 542

,0
ESTADOCIVIL,
fem div/cas,308
masculino casado/viuvo,92
masculino div/sep,50
masculino solteiro,542


In [ ]:
x['ESTADOCIVIL'].fillna('masculino solteiro', inplace=True)
x.isnull().sum() #e aqui nós zeramos igual no salto atual

,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,0
PROPOSITO,0


In [ ]:
#vericando valores fora do padrão

desv = x['SALDO_ATUAL'].std()
desv

685936688.9820064

In [ ]:
x.loc[x['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL']

,SALDO_ATUAL
127,2.541111e+09
160,2.154441e+10


In [ ]:
mediana = x['SALDO_ATUAL'].median()
mediana

2323.0

In [ ]:
x.loc[x['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL'] = mediana
x.loc[x['SALDO_ATUAL']>= 2 * desv,]

#retiramos os outliers para nao ter problemas no processamento

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO


In [ ]:
#agora retiraremos items escritos errados

agrupado = x.groupby(['PROPOSITO']).size()
agrupado

,0
PROPOSITO,
Eletrodomésticos,12
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,12
qualificação,9


In [ ]:
x.loc[x['PROPOSITO']=='Eletrodomésticos','PROPOSITO'] = 'outros'
x.loc[x['PROPOSITO']=='qualificação','PROPOSITO'] = 'outros'
agrupado = x.groupby(['PROPOSITO']).size()
agrupado


,0
PROPOSITO,
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,33
radio/tv,280


In [ ]:
x['DATA']

,DATA
0,01/01/2019
1,01/01/2020
2,02/01/2020
3,02/01/2019
4,03/01/2018
...,...
995,29/06/2018
996,30/06/2018
997,03/07/2018
998,04/07/2019


In [ ]:
x['DATA'] = pd.to_datetime(x['DATA'], format='%d/%m/%Y')

In [ ]:
x['DATA']

,DATA
0,2019-01-01
1,2020-01-01
2,2020-01-02
3,2019-01-02
4,2018-01-03
...,...
995,2018-06-29
996,2018-06-30
997,2018-07-03
998,2019-07-04


In [ ]:
x['ANO'] = x['DATA'].dt.year
x['MES'] = x['DATA'].dt.month
x['DIASEMANA'] = x['DATA'].dt.day_name()


In [ ]:
x['DIASEMANA']

,DIASEMANA
0,Tuesday
1,Wednesday
2,Thursday
3,Wednesday
4,Wednesday
...,...
995,Friday
996,Saturday
997,Tuesday
998,Thursday


In [ ]:
#label encoder irá incluir valor numericos para algumas informações categoricas
 x['ESTADOCIVIL'].unique()

IndentationError: unexpected indent (<ipython-input-50-e45143d8483d>, line 2)

In [ ]:
x['PROPOSITO'].unique()

array(['radio/tv', 'educação', 'mobilia/equipamento', 'carro novo',
       'carro usado', 'negócios', 'outros', 'obras'], dtype=object)

In [ ]:
x['DIASEMANA'].unique()

array(['Tuesday', 'Wednesday', 'Thursday', 'Saturday', 'Sunday', 'Monday',
       'Friday'], dtype=object)

In [ ]:
labelencoder1 = LabelEncoder()
x['DIASEMANA'] = labelencoder1.fit_transform(x['DIASEMANA'])
x['PROPOSITO'] = labelencoder1.fit_transform(x['PROPOSITO'])

In [ ]:
x.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,PROPOSITOS,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,masculino solteiro,7,NaN,2019,1,5
1,5951.0,2,22,nenhum,2020-01-01,fem div/cas,7,NaN,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,masculino solteiro,2,NaN,2020,1,4
3,7882.0,4,45,nenhum,2019-01-02,masculino solteiro,3,NaN,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,masculino solteiro,0,NaN,2018,1,6


In [ ]:
outros = x['OUTROSPLANOSPGTO'].unique()
outros

array(['nenhum', 'banco', 'stores'], dtype=object)

In [ ]:
z =pd.get_dummies(x['OUTROSPLANOSPGTO'], prefix = 'OUTROS')
z

,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False
...,...,...,...
995,False,True,False
996,False,True,False
997,False,True,False
998,False,True,False


In [ ]:
x

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,PROPOSITOS,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,masculino solteiro,7,NaN,2019,1,5
1,5951.0,2,22,nenhum,2020-01-01,fem div/cas,7,NaN,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,masculino solteiro,2,NaN,2020,1,4
3,7882.0,4,45,nenhum,2019-01-02,masculino solteiro,3,NaN,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,masculino solteiro,0,NaN,2018,1,6
...,...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,fem div/cas,3,NaN,2018,6,0
996,3857.0,4,40,nenhum,2018-06-30,masculino div/sep,1,NaN,2018,6,2
997,804.0,4,38,nenhum,2018-07-03,masculino solteiro,7,NaN,2018,7,5
998,1845.0,4,23,nenhum,2019-07-04,masculino solteiro,7,NaN,2019,7,4


In [ ]:
sc = StandardScaler()
m = sc.fit_transform(x.iloc[:,0:3])
m

array([[-0.74551643,  1.04698668,  1.6392759 ],
       [ 0.95774038, -0.76597727, -0.74024139],
       [-0.41533679,  0.14050471,  0.68746898],
       ...,
       [-0.87552244,  1.04698668,  0.1058092 ],
       [-0.50473818,  1.04698668, -0.68736323],
       [ 0.46799171,  1.04698668, -0.47585058]])

In [ ]:
x = pd.concat([x,z,pd.DataFrame(m,columns=['SALDO_ATUAL_N','RESIDENCIADESDE_N', 'IDADE_N'])],axis=1)

In [ ]:
x.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,PROPOSITOS,ANO,MES,DIASEMANA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCIADESDE_N,IDADE_N
0,1169.0,4,67,nenhum,2019-01-01,masculino solteiro,7,NaN,2019,1,5,False,True,False,-0.745516,1.046987,1.639276
1,5951.0,2,22,nenhum,2020-01-01,fem div/cas,7,NaN,2020,1,6,False,True,False,0.957740,-0.765977,-0.740241
2,2096.0,3,49,nenhum,2020-01-02,masculino solteiro,2,NaN,2020,1,4,False,True,False,-0.415337,0.140505,0.687469
3,7882.0,4,45,nenhum,2019-01-02,masculino solteiro,3,NaN,2019,1,6,False,True,False,1.645526,1.046987,0.475956
4,4870.0,4,53,nenhum,2018-01-03,masculino solteiro,0,NaN,2018,1,6,False,True,False,0.572709,1.046987,0.898982


In [ ]:
x.drop(columns=['SALDO_ATUAL','RESIDENCIADESDE','OUTROSPLANOSPGTO','DATA','OUTROS_banco'],inplace=True)

In [ ]:
x

,IDADE,ESTADOCIVIL,PROPOSITO,PROPOSITOS,ANO,MES,DIASEMANA,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCIADESDE_N,IDADE_N
0,67,masculino solteiro,7,NaN,2019,1,5,True,False,-0.745516,1.046987,1.639276
1,22,fem div/cas,7,NaN,2020,1,6,True,False,0.957740,-0.765977,-0.740241
2,49,masculino solteiro,2,NaN,2020,1,4,True,False,-0.415337,0.140505,0.687469
3,45,masculino solteiro,3,NaN,2019,1,6,True,False,1.645526,1.046987,0.475956
4,53,masculino solteiro,0,NaN,2018,1,6,True,False,0.572709,1.046987,0.898982
...,...,...,...,...,...,...,...,...,...,...,...,...
995,31,fem div/cas,3,NaN,2018,6,0,True,False,-0.543562,1.046987,-0.264338
996,40,masculino div/sep,1,NaN,2018,6,2,True,False,0.211898,1.046987,0.211566
997,38,masculino solteiro,7,NaN,2018,7,5,True,False,-0.875522,1.046987,0.105809
998,23,masculino solteiro,7,NaN,2019,7,4,True,False,-0.504738,1.046987,-0.687363
